# Python & FastAPI Interview Q&A
## For 5-6 Years Experience | Backend & Architecture Focus

---

# SECTION 1: Python Internals & Advanced Concepts

## 1.1 Memory Management & Object Model

**Q1: Explain Python's memory management and how the GIL affects multi-threaded FastAPI applications.**

Python uses reference counting as its primary memory management strategy, combined with a cyclic garbage collector. The GIL (Global Interpreter Lock) ensures only one thread executes Python bytecode at a time.

```python
import sys
import gc

x = [1, 2, 3]
y = x  # ref count = 2
print(sys.getrefcount(x))  # 3 (getrefcount itself adds 1)

del y   # ref count drops to 1
gc.collect()  # force cyclic GC
```

**In FastAPI context:** FastAPI uses `asyncio` (single-threaded event loop), so the GIL is largely irrelevant for I/O-bound work. For CPU-bound tasks, use `ProcessPoolExecutor`.

```python
from fastapi import FastAPI
from concurrent.futures import ProcessPoolExecutor
import asyncio

app = FastAPI()
executor = ProcessPoolExecutor()

def cpu_heavy(n: int) -> int:
    return sum(i * i for i in range(n))

@app.get("/compute/{n}")
async def compute(n: int):
    loop = asyncio.get_event_loop()
    result = await loop.run_in_executor(executor, cpu_heavy, n)
    return {"result": result}
```

---

**Q2: What are descriptors in Python? Show a practical use case.**

Descriptors implement `__get__`, `__set__`, or `__delete__`. They power `@property`, `classmethod`, `staticmethod`, and ORM fields.

```python
class ValidatedField:
    def __set_name__(self, owner, name):
        self.name = name
        self.private = f"_{name}"

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, self.private, None)

    def __set__(self, obj, value):
        if not isinstance(value, (int, float)) or value < 0:
            raise ValueError(f"{self.name} must be a non-negative number")
        setattr(obj, self.private, value)

class Product:
    price = ValidatedField()
    stock = ValidatedField()

    def __init__(self, price, stock):
        self.price = price
        self.stock = stock

p = Product(99.9, 10)
p.price = -5  # raises ValueError
```

---

**Q3: Explain `__slots__` and when you'd use them in a high-throughput FastAPI service.**

```python
# Without __slots__: each instance gets a __dict__ (~200+ bytes overhead)
class RequestMetadata:
    def __init__(self, user_id, ip, timestamp):
        self.user_id = user_id
        self.ip = ip
        self.timestamp = timestamp

# With __slots__: no __dict__, ~3x faster attribute access, less memory
class RequestMetadataSlotted:
    __slots__ = ('user_id', 'ip', 'timestamp')

    def __init__(self, user_id, ip, timestamp):
        self.user_id = user_id
        self.ip = ip
        self.timestamp = timestamp

import sys
a = RequestMetadata("u1", "127.0.0.1", 1700000000)
b = RequestMetadataSlotted("u1", "127.0.0.1", 1700000000)
print(sys.getsizeof(a.__dict__))  # ~200 bytes
# b has no __dict__ - use for millions of short-lived objects
```

**Use when:** processing millions of short-lived request/event objects, data pipeline records, or parser tokens.

---

## 1.2 Async Python Deep Dive

**Q4: What is the difference between `asyncio.gather`, `asyncio.wait`, and `TaskGroup`? When does each shine?**

```python
import asyncio

async def fetch(id: int, delay: float):
    await asyncio.sleep(delay)
    if id == 3:
        raise ValueError("fetch failed")
    return f"data_{id}"

# gather: collect all results, fail-fast by default
async def use_gather():
    results = await asyncio.gather(
        fetch(1, 0.1), fetch(2, 0.2), fetch(3, 0.05),
        return_exceptions=True  # don't cancel others on failure
    )
    return [r for r in results if not isinstance(r, Exception)]

# wait: fine-grained control - get results as they complete
async def use_wait():
    tasks = {asyncio.create_task(fetch(i, 0.1 * i)) for i in range(1, 4)}
    done, pending = await asyncio.wait(tasks, return_when=asyncio.FIRST_EXCEPTION)
    for t in pending:
        t.cancel()
    return [t.result() for t in done if not t.exception()]

# TaskGroup (Python 3.11+): structured concurrency, clean cancellation
async def use_taskgroup():
    results = []
    async with asyncio.TaskGroup() as tg:
        tasks = [tg.create_task(fetch(i, 0.1)) for i in [1, 2, 4]]
    return [t.result() for t in tasks]
```

| | `gather` | `wait` | `TaskGroup` |
|---|---|---|---|
| Cancel on error | optional | manual | automatic |
| Partial results | yes | yes | no |
| Python version | 3.4+ | 3.4+ | 3.11+ |

---

**Q5: Explain `async` context managers and generators. Build a database transaction context manager.**

```python
from contextlib import asynccontextmanager
from typing import AsyncGenerator

class AsyncDBConnection:
    async def execute(self, query: str): ...
    async def commit(self): ...
    async def rollback(self): ...
    async def close(self): ...

@asynccontextmanager
async def db_transaction(pool) -> AsyncGenerator[AsyncDBConnection, None]:
    conn = await pool.acquire()
    try:
        yield conn
        await conn.commit()
    except Exception:
        await conn.rollback()
        raise
    finally:
        await pool.release(conn)

# Usage in FastAPI endpoint
@app.post("/transfer")
async def transfer(from_id: int, to_id: int, amount: float):
    async with db_transaction(db_pool) as conn:
        await conn.execute(f"UPDATE accounts SET balance = balance - {amount} WHERE id = {from_id}")
        await conn.execute(f"UPDATE accounts SET balance = balance + {amount} WHERE id = {to_id}")
    return {"status": "transferred"}
```

---

**Q6: What are Python protocol classes and how do they differ from ABCs?**

```python
from typing import Protocol, runtime_checkable

# ABC: nominal subtyping (must explicitly inherit)
from abc import ABC, abstractmethod

class Repository(ABC):
    @abstractmethod
    async def get(self, id: int): ...

# Protocol: structural subtyping (duck typing with type hints)
@runtime_checkable
class CacheBackend(Protocol):
    async def get(self, key: str) -> bytes | None: ...
    async def set(self, key: str, value: bytes, ttl: int) -> None: ...
    async def delete(self, key: str) -> None: ...

# RedisCache doesn't inherit CacheBackend — but satisfies the protocol
class RedisCache:
    async def get(self, key: str) -> bytes | None:
        return await self.redis.get(key)

    async def set(self, key: str, value: bytes, ttl: int) -> None:
        await self.redis.setex(key, ttl, value)

    async def delete(self, key: str) -> None:
        await self.redis.delete(key)

cache: CacheBackend = RedisCache()  # type-checks correctly
print(isinstance(cache, CacheBackend))  # True (runtime_checkable)
```

**Key difference:** ABCs require explicit inheritance; Protocols check structural compatibility at type-check time (and optionally at runtime).

---

## 1.3 Generators, Itertools & Functional Python

**Q7: Implement a memory-efficient data pipeline using generators for processing large datasets in a FastAPI background task.**

```python
from typing import Iterator, Generator
import csv, io

def read_csv_chunks(file_bytes: bytes, chunk_size: int = 1000) -> Iterator[list[dict]]:
    reader = csv.DictReader(io.StringIO(file_bytes.decode()))
    chunk = []
    for row in reader:
        chunk.append(row)
        if len(chunk) >= chunk_size:
            yield chunk
            chunk = []
    if chunk:
        yield chunk

def validate_rows(chunks: Iterator[list[dict]]) -> Generator[dict, None, None]:
    for chunk in chunks:
        for row in chunk:
            if row.get("email") and "@" in row["email"]:
                yield row

def transform(rows: Generator) -> Generator[dict, None, None]:
    for row in rows:
        yield {
            "email": row["email"].lower().strip(),
            "name": row.get("name", "").title(),
        }

async def process_upload(file_bytes: bytes, db):
    pipeline = transform(validate_rows(read_csv_chunks(file_bytes)))
    batch = []
    async with db_transaction(db) as conn:
        for record in pipeline:
            batch.append(record)
            if len(batch) >= 500:
                await conn.execute_many("INSERT INTO users ...", batch)
                batch = []
        if batch:
            await conn.execute_many("INSERT INTO users ...", batch)
```

---

# SECTION 2: FastAPI Internals & Patterns

## 2.1 Dependency Injection System

**Q8: How does FastAPI's DI system work under the hood? What are the different dependency scopes?**

FastAPI uses `Depends()` to build a dependency graph resolved per-request. Dependencies can be functions, classes, or generators.

```python
from fastapi import FastAPI, Depends, Request
from functools import lru_cache

# 1. Function dependency
async def get_db():
    db = SessionLocal()
    try:
        yield db          # generator = teardown guaranteed
    finally:
        db.close()

# 2. Class dependency (callable class)
class Paginator:
    def __init__(self, page: int = 1, size: int = 20):
        self.offset = (page - 1) * size
        self.limit = min(size, 100)  # cap page size

# 3. Cached / app-lifetime dependency
@lru_cache
def get_settings():
    return Settings()  # loaded once, reused forever

# 4. Dependency with sub-dependency
async def get_current_user(
    token: str = Depends(oauth2_scheme),
    db: Session = Depends(get_db),
    settings: Settings = Depends(get_settings)
):
    payload = jwt.decode(token, settings.secret_key)
    return await db.get(User, payload["sub"])

@app.get("/profile")
async def profile(user = Depends(get_current_user)):
    return user
```

---

**Q9: Build a role-based access control (RBAC) dependency chain.**

```python
from enum import Enum
from fastapi import HTTPException, status

class Role(str, Enum):
    VIEWER = "viewer"
    EDITOR = "editor"
    ADMIN = "admin"

ROLE_HIERARCHY = {Role.VIEWER: 0, Role.EDITOR: 1, Role.ADMIN: 2}

def require_role(minimum_role: Role):
    async def checker(current_user = Depends(get_current_user)):
        user_level = ROLE_HIERARCHY.get(current_user.role, -1)
        required_level = ROLE_HIERARCHY[minimum_role]
        if user_level < required_level:
            raise HTTPException(
                status_code=status.HTTP_403_FORBIDDEN,
                detail=f"Requires {minimum_role} role or higher"
            )
        return current_user
    return checker

# Usage
@app.delete("/resource/{id}")
async def delete_resource(
    id: int,
    user = Depends(require_role(Role.ADMIN))
):
    ...

@app.put("/resource/{id}")
async def update_resource(
    id: int,
    user = Depends(require_role(Role.EDITOR))
):
    ...
```

---

## 2.2 Pydantic V2 Deep Dive

**Q10: What changed between Pydantic V1 and V2? Show migration pitfalls.**

```python
from pydantic import BaseModel, field_validator, model_validator, Field
from pydantic import ConfigDict
from typing import Annotated

# V1 style (deprecated)
# class User(BaseModel):
#     class Config:
#         orm_mode = True
#     @validator('email')
#     def check_email(cls, v): ...

# V2 style
class UserCreate(BaseModel):
    model_config = ConfigDict(from_attributes=True)  # replaces orm_mode

    name: str = Field(min_length=2, max_length=50)
    email: Annotated[str, Field(pattern=r'^[\w.+-]+@[\w-]+\.[a-z]{2,}$')]
    age: int = Field(ge=0, le=120)
    password: str

    @field_validator('email')           # replaces @validator
    @classmethod
    def normalize_email(cls, v: str) -> str:
        return v.lower().strip()

    @model_validator(mode='after')      # whole-model validation
    def check_age_for_terms(self) -> 'UserCreate':
        if self.age < 18:
            raise ValueError("Must be 18+ to register")
        return self

# V2 is 5-50x faster than V1 (Rust core via pydantic-core)
# Key breaking changes:
# - .dict() → .model_dump()
# - .json() → .model_dump_json()
# - .parse_obj() → .model_validate()
# - @validator → @field_validator (must be @classmethod)
# - orm_mode → from_attributes
```

---

**Q11: Custom Pydantic types and serializers for domain-driven design.**

```python
from pydantic import BaseModel, GetCoreSchemaHandler
from pydantic_core import core_schema
from typing import Any

class Money:
    """Value object: immutable, equality by value"""
    def __init__(self, amount: int, currency: str = "USD"):
        self.amount = amount      # store as cents
        self.currency = currency

    @classmethod
    def __get_pydantic_core_schema__(cls, source, handler: GetCoreSchemaHandler):
        return core_schema.no_info_plain_validator_function(cls._validate)

    @classmethod
    def _validate(cls, v: Any) -> 'Money':
        if isinstance(v, cls):
            return v
        if isinstance(v, dict):
            return cls(v['amount'], v.get('currency', 'USD'))
        if isinstance(v, (int, float)):
            return cls(int(v * 100))
        raise ValueError(f"Cannot convert {type(v)} to Money")

    def __repr__(self):
        return f"${self.amount / 100:.2f} {self.currency}"

class Order(BaseModel):
    id: int
    total: Money
    discount: Money = Money(0)

order = Order(id=1, total={"amount": 9999, "currency": "USD"})
print(order.total)  # $99.99 USD
```

---

## 2.3 Middleware, Lifespan & Background Tasks

**Q12: Implement production-grade middleware for request tracing and rate limiting.**

```python
import time, uuid
from fastapi import FastAPI, Request, Response
from starlette.middleware.base import BaseHTTPMiddleware

class TracingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        request_id = request.headers.get("X-Request-ID", str(uuid.uuid4()))
        request.state.request_id = request_id
        start = time.perf_counter()

        response: Response = await call_next(request)

        duration_ms = (time.perf_counter() - start) * 1000
        response.headers["X-Request-ID"] = request_id
        response.headers["X-Response-Time"] = f"{duration_ms:.2f}ms"

        # Structured logging
        logger.info("request_completed", extra={
            "request_id": request_id,
            "method": request.method,
            "path": request.url.path,
            "status": response.status_code,
            "duration_ms": round(duration_ms, 2),
        })
        return response

# Lifespan: replaces on_event("startup"/"shutdown")
from contextlib import asynccontextmanager

@asynccontextmanager
async def lifespan(app: FastAPI):
    # startup
    app.state.redis = await create_redis_pool()
    app.state.db_pool = await create_db_pool()
    yield
    # shutdown
    await app.state.redis.aclose()
    await app.state.db_pool.close()

app = FastAPI(lifespan=lifespan)
app.add_middleware(TracingMiddleware)
```

---

**Q13: What's the difference between `BackgroundTasks` and Celery/ARQ for async jobs?**

```python
# BackgroundTasks: in-process, fire-and-forget, no persistence
from fastapi import BackgroundTasks

async def send_email(to: str, subject: str):
    await email_client.send(to, subject)   # runs after response sent

@app.post("/register")
async def register(user: UserCreate, bg: BackgroundTasks):
    db_user = await create_user(user)
    bg.add_task(send_email, db_user.email, "Welcome!")
    return db_user  # response sent immediately

# ARQ: Redis-backed, persistent, retriable, distributed
import arq

async def process_report(ctx, report_id: int):
    """This runs in a separate worker process"""
    data = await fetch_report_data(report_id)
    pdf = await generate_pdf(data)
    await upload_to_s3(pdf)

@app.post("/reports/{id}/generate")
async def trigger_report(id: int, redis=Depends(get_redis)):
    job = await redis.enqueue_job('process_report', id)
    return {"job_id": job.job_id}
```

| | BackgroundTasks | ARQ/Celery |
|---|---|---|
| Persistence | No | Yes |
| Retry on crash | No | Yes |
| Distributed workers | No | Yes |
| Overhead | Zero | Redis/broker |
| Use for | emails, webhooks | heavy jobs, pipelines |

---

# SECTION 3: Architecture Patterns with FastAPI

## 3.1 Event-Driven Architecture (EDA)

**Q14: How do you implement an event-driven pattern with FastAPI as a producer/consumer using Kafka?**

```python
# Producer side (FastAPI)
from aiokafka import AIOKafkaProducer
import json

class EventBus:
    def __init__(self):
        self._producer: AIOKafkaProducer | None = None

    async def start(self, bootstrap_servers: str):
        self._producer = AIOKafkaProducer(
            bootstrap_servers=bootstrap_servers,
            value_serializer=lambda v: json.dumps(v).encode()
        )
        await self._producer.start()

    async def publish(self, topic: str, event: dict, key: str | None = None):
        await self._producer.send_and_wait(
            topic,
            value=event,
            key=key.encode() if key else None
        )

event_bus = EventBus()

@app.post("/orders")
async def create_order(order: OrderCreate):
    db_order = await save_order(order)
    await event_bus.publish(
        topic="order.created",
        event={"order_id": db_order.id, "user_id": order.user_id, "total": order.total},
        key=str(db_order.id)   # partition by order_id
    )
    return db_order

# Consumer side (separate service or worker)
from aiokafka import AIOKafkaConsumer

async def order_consumer():
    consumer = AIOKafkaConsumer(
        "order.created",
        bootstrap_servers="localhost:9092",
        group_id="inventory-service",
        value_deserializer=lambda b: json.loads(b.decode())
    )
    await consumer.start()
    async for msg in consumer:
        await handle_order_created(msg.value)
```

---

**Q15: Implement the Outbox Pattern to guarantee event delivery.**

```python
# The problem: DB write + Kafka publish can fail independently
# Solution: write event to DB (same transaction), poll and publish

class OutboxEvent(Base):
    __tablename__ = "outbox"
    id = Column(UUID, primary_key=True, default=uuid4)
    topic = Column(String, nullable=False)
    payload = Column(JSONB, nullable=False)
    created_at = Column(DateTime, default=datetime.utcnow)
    published_at = Column(DateTime, nullable=True)

async def create_order_with_outbox(order: OrderCreate, db: AsyncSession):
    async with db.begin():
        db_order = Order(**order.dict())
        db.add(db_order)

        # Write event in same transaction - atomicity guaranteed
        outbox_event = OutboxEvent(
            topic="order.created",
            payload={"order_id": str(db_order.id), "total": order.total}
        )
        db.add(outbox_event)
    return db_order

# Background poller (runs in worker or separate process)
async def outbox_relay(db: AsyncSession, producer: AIOKafkaProducer):
    while True:
        async with db.begin():
            events = await db.execute(
                select(OutboxEvent)
                .where(OutboxEvent.published_at.is_(None))
                .order_by(OutboxEvent.created_at)
                .limit(100)
                .with_for_update(skip_locked=True)  # concurrent-safe
            )
            for event in events.scalars():
                await producer.send_and_wait(event.topic, event.payload)
                event.published_at = datetime.utcnow()
        await asyncio.sleep(1)
```

---

## 3.2 CQRS (Command Query Responsibility Segregation)

**Q16: Implement CQRS with separate read/write models in FastAPI.**

```python
# Write model (normalized, consistent)
class OrderWriteModel(Base):
    __tablename__ = "orders"
    id = Column(UUID, primary_key=True)
    user_id = Column(Integer, ForeignKey("users.id"))
    status = Column(String)
    created_at = Column(DateTime)

# Read model (denormalized, optimized for query)
class OrderReadModel(Base):
    __tablename__ = "orders_view"
    id = Column(UUID, primary_key=True)
    user_email = Column(String)     # denormalized
    user_name = Column(String)
    status = Column(String)
    item_count = Column(Integer)    # pre-aggregated
    total_amount = Column(Numeric)

# Commands (writes)
class CreateOrderCommand(BaseModel):
    user_id: int
    items: list[OrderItem]

async def handle_create_order(cmd: CreateOrderCommand, write_db):
    order = OrderWriteModel(id=uuid4(), user_id=cmd.user_id, status="pending")
    write_db.add(order)
    await write_db.commit()
    await event_bus.publish("order.created", {"order_id": str(order.id)})
    return order.id

# Queries (reads from read model)
async def get_user_orders(user_id: int, read_db) -> list[OrderReadModel]:
    result = await read_db.execute(
        select(OrderReadModel).where(OrderReadModel.user_id == user_id)
    )
    return result.scalars().all()

# Projector: keeps read model in sync
async def on_order_created(event: dict, write_db, read_db):
    order = await write_db.get(OrderWriteModel, event["order_id"])
    user = await write_db.get(User, order.user_id)
    read_model = OrderReadModel(
        id=order.id, user_email=user.email,
        user_name=user.name, status=order.status
    )
    read_db.add(read_model)
    await read_db.commit()
```

---

## 3.3 GraphQL with FastAPI (Strawberry)

**Q17: Integrate GraphQL with FastAPI using Strawberry, including DataLoader to solve N+1.**

```python
import strawberry
from strawberry.fastapi import GraphQLRouter
from strawberry.dataloader import DataLoader
from typing import List

@strawberry.type
class UserType:
    id: int
    name: str
    email: str

@strawberry.type
class OrderType:
    id: strawberry.ID
    total: float
    user: UserType

# DataLoader batches N individual DB calls into 1
async def load_users_by_ids(ids: list[int]) -> list[UserType]:
    # ONE query for all ids instead of N queries
    users = await db.execute(select(User).where(User.id.in_(ids)))
    user_map = {u.id: UserType(id=u.id, name=u.name, email=u.email)
                for u in users.scalars()}
    return [user_map.get(id) for id in ids]

def get_context():
    return {"user_loader": DataLoader(load_fn=load_users_by_ids)}

@strawberry.type
class Query:
    @strawberry.field
    async def orders(self, info: strawberry.types.Info) -> List[OrderType]:
        orders = await fetch_all_orders()
        return [
            OrderType(
                id=o.id,
                total=o.total,
                # DataLoader deduplicates + batches these
                user=await info.context["user_loader"].load(o.user_id)
            ) for o in orders
        ]

schema = strawberry.Schema(query=Query)
graphql_router = GraphQLRouter(schema, context_getter=get_context)
app.include_router(graphql_router, prefix="/graphql")
```

---

## 3.4 gRPC with FastAPI

**Q18: How do you expose both REST and gRPC from a FastAPI service?**

```python
# orders.proto
"""
syntax = "proto3";
service OrderService {
  rpc CreateOrder (CreateOrderRequest) returns (OrderResponse);
  rpc StreamOrders (OrderFilter) returns (stream OrderResponse);
}
"""

import grpc
from concurrent import futures
import asyncio

# gRPC server runs alongside FastAPI (different port)
class OrderServicer(orders_pb2_grpc.OrderServiceServicer):
    async def CreateOrder(self, request, context):
        order = await create_order_internal(
            user_id=request.user_id,
            items=list(request.items)
        )
        return orders_pb2.OrderResponse(
            order_id=str(order.id),
            status=order.status
        )

    async def StreamOrders(self, request, context):
        async for order in watch_orders(user_id=request.user_id):
            yield orders_pb2.OrderResponse(
                order_id=str(order.id),
                status=order.status
            )

async def serve_grpc():
    server = grpc.aio.server()
    orders_pb2_grpc.add_OrderServiceServicer_to_server(OrderServicer(), server)
    server.add_insecure_port("[::]:50051")
    await server.start()
    await server.wait_for_termination()

# Run both in same process
@asynccontextmanager
async def lifespan(app: FastAPI):
    grpc_task = asyncio.create_task(serve_grpc())
    yield
    grpc_task.cancel()

app = FastAPI(lifespan=lifespan)

# REST endpoint calls same internal logic
@app.post("/orders")
async def rest_create_order(order: OrderCreate):
    return await create_order_internal(order.user_id, order.items)
```

---

## 3.5 WebSocket Architecture

**Q19: Build a scalable WebSocket hub with pub/sub using Redis.**

```python
from fastapi import WebSocket, WebSocketDisconnect
import json

class ConnectionManager:
    def __init__(self, redis):
        self.redis = redis
        self.local_connections: dict[str, set[WebSocket]] = {}

    async def connect(self, ws: WebSocket, channel: str):
        await ws.accept()
        self.local_connections.setdefault(channel, set()).add(ws)
        # Subscribe this pod's listener to Redis channel
        await self.redis.subscribe(f"ws:{channel}")

    async def disconnect(self, ws: WebSocket, channel: str):
        self.local_connections[channel].discard(ws)

    async def broadcast_to_channel(self, channel: str, message: dict):
        # Publish to Redis → all pods receive it
        await self.redis.publish(f"ws:{channel}", json.dumps(message))

    async def listen_redis(self):
        """Runs as background task - relay Redis messages to local WS clients"""
        pubsub = self.redis.pubsub()
        async for message in pubsub.listen():
            if message["type"] == "message":
                channel = message["channel"].decode().removeprefix("ws:")
                data = json.loads(message["data"])
                dead = set()
                for ws in self.local_connections.get(channel, set()):
                    try:
                        await ws.send_json(data)
                    except Exception:
                        dead.add(ws)
                self.local_connections[channel] -= dead

manager = ConnectionManager(redis)

@app.websocket("/ws/{room_id}")
async def websocket_endpoint(ws: WebSocket, room_id: str):
    await manager.connect(ws, room_id)
    try:
        while True:
            data = await ws.receive_json()
            await manager.broadcast_to_channel(room_id, {
                "sender": data["user_id"],
                "message": data["text"]
            })
    except WebSocketDisconnect:
        await manager.disconnect(ws, room_id)
```

---

# SECTION 4: Testing & Code Quality

## 4.1 Testing FastAPI Applications

**Q20: Write comprehensive tests using `pytest` and `httpx` with dependency overrides.**

```python
import pytest
from httpx import AsyncClient, ASGITransport
from unittest.mock import AsyncMock, patch

# Fixtures
@pytest.fixture
def mock_db():
    return AsyncMock()

@pytest.fixture
async def client(mock_db):
    app.dependency_overrides[get_db] = lambda: mock_db
    async with AsyncClient(
        transport=ASGITransport(app=app), base_url="http://test"
    ) as ac:
        yield ac
    app.dependency_overrides.clear()

# Test with mocked dependencies
@pytest.mark.asyncio
async def test_create_user_success(client, mock_db):
    mock_db.execute.return_value.scalar_one_or_none.return_value = None  # no dup
    mock_db.add = AsyncMock()
    mock_db.commit = AsyncMock()

    response = await client.post("/users", json={
        "name": "Alice", "email": "alice@example.com", "password": "secret123", "age": 25
    })

    assert response.status_code == 201
    assert response.json()["email"] == "alice@example.com"

@pytest.mark.asyncio
async def test_create_user_duplicate_email(client, mock_db):
    existing_user = User(id=1, email="alice@example.com")
    mock_db.execute.return_value.scalar_one_or_none.return_value = existing_user

    response = await client.post("/users", json={
        "name": "Bob", "email": "alice@example.com", "password": "pass", "age": 30
    })
    assert response.status_code == 409

# Parametrize for edge cases
@pytest.mark.parametrize("payload,expected_status", [
    ({"name": "A", "email": "bad", "age": 25}, 422),    # bad email
    ({"name": "Alice", "email": "a@b.com", "age": -1}, 422),  # negative age
    ({"name": "Alice", "email": "a@b.com", "age": 16}, 422),  # underage
])
@pytest.mark.asyncio
async def test_user_validation(client, payload, expected_status):
    response = await client.post("/users", json=payload)
    assert response.status_code == expected_status
```

---

**Q21: What's the difference between mocking, stubbing, and faking? Show each in a FastAPI test.**

```python
# STUB: hardcoded return value, no logic
stub_cache = AsyncMock()
stub_cache.get.return_value = b'{"cached": true}'

# MOCK: like stub but also verifies calls were made
mock_email = AsyncMock()
await service.register_user(user_data)
mock_email.send.assert_called_once_with("welcome@example.com", ...)

# FAKE: working implementation, not production-grade
class FakeCache:
    """In-memory cache for tests — real logic, no Redis"""
    def __init__(self):
        self._store: dict[str, bytes] = {}

    async def get(self, key: str) -> bytes | None:
        return self._store.get(key)

    async def set(self, key: str, value: bytes, ttl: int = 0) -> None:
        self._store[key] = value

    async def delete(self, key: str) -> None:
        self._store.pop(key, None)

# Using fake in tests
@pytest.fixture
def fake_cache():
    return FakeCache()

@pytest.fixture
async def client(fake_cache):
    app.dependency_overrides[get_cache] = lambda: fake_cache
    async with AsyncClient(...) as ac:
        yield ac
```

---

# SECTION 5: Database Patterns & SQLAlchemy

## 5.1 Advanced SQLAlchemy

**Q22: What is the Unit of Work pattern and how does SQLAlchemy implement it?**

```python
from sqlalchemy.ext.asyncio import AsyncSession, create_async_engine, async_sessionmaker

engine = create_async_engine("postgresql+asyncpg://...", pool_size=10, max_overflow=20)
AsyncSessionLocal = async_sessionmaker(engine, expire_on_commit=False)

# SQLAlchemy Session IS the Unit of Work
# - Tracks all changes (identity map)
# - Flushes them as a single batch
# - Rolls back everything if any step fails

async def transfer_inventory(from_id: int, to_id: int, qty: int):
    async with AsyncSessionLocal() as session:
        async with session.begin():  # UoW boundary
            src = await session.get(InventoryItem, from_id, with_for_update=True)
            dst = await session.get(InventoryItem, to_id, with_for_update=True)

            if src.quantity < qty:
                raise ValueError("Insufficient inventory")

            src.quantity -= qty  # tracked as dirty
            dst.quantity += qty  # tracked as dirty

            # session.begin() auto-commits or rolls back
            # Both changes flush together in one transaction
```

---

**Q23: Explain N+1 query problem and fix it with SQLAlchemy.**

```python
from sqlalchemy.orm import selectinload, joinedload

# BAD: N+1 - 1 query for orders + N queries for each user
async def bad_get_orders(db: AsyncSession):
    orders = (await db.execute(select(Order))).scalars().all()
    return [{"id": o.id, "user": o.user.name} for o in orders]  # lazy load fires!

# GOOD option 1: joinedload (single JOIN query)
async def good_get_orders_join(db: AsyncSession):
    result = await db.execute(
        select(Order).options(joinedload(Order.user))
    )
    orders = result.unique().scalars().all()
    return [{"id": o.id, "user": o.user.name} for o in orders]

# GOOD option 2: selectinload (2 queries, better for collections)
async def good_get_orders_select_in(db: AsyncSession):
    result = await db.execute(
        select(User).options(selectinload(User.orders))
    )
    users = result.scalars().all()
    return users   # orders pre-loaded in second IN query

# Use joinedload for many-to-one (Order → User)
# Use selectinload for one-to-many (User → Orders) to avoid row duplication
```

---

# SECTION 6: Performance, Observability & Production

## 6.1 Caching Strategies

**Q24: Implement a multi-level cache with cache-aside pattern.**

```python
from functools import wraps
import pickle, hashlib

def cache(ttl: int = 300, key_prefix: str = ""):
    def decorator(func):
        @wraps(func)
        async def wrapper(*args, **kwargs):
            # Build deterministic cache key
            key_data = f"{key_prefix}:{func.__name__}:{args}:{sorted(kwargs.items())}"
            cache_key = hashlib.md5(key_data.encode()).hexdigest()

            # L1: In-process dict (ultra-fast, per-worker)
            if cache_key in _local_cache:
                return _local_cache[cache_key]

            # L2: Redis (shared across workers)
            redis = get_redis()
            cached = await redis.get(cache_key)
            if cached:
                result = pickle.loads(cached)
                _local_cache[cache_key] = result
                return result

            # Cache miss: call function and populate
            result = await func(*args, **kwargs)
            await redis.setex(cache_key, ttl, pickle.dumps(result))
            _local_cache[cache_key] = result
            return result
        return wrapper
    return decorator

_local_cache: dict = {}

@cache(ttl=60, key_prefix="product")
async def get_product(product_id: int) -> dict:
    return await db.fetch_product(product_id)
```

---

## 6.2 Observability

**Q25: Implement structured logging and OpenTelemetry tracing in FastAPI.**

```python
import structlog
from opentelemetry import trace
from opentelemetry.instrumentation.fastapi import FastAPIInstrumentor
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter

# Setup tracing
provider = TracerProvider()
provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter()))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer(__name__)

FastAPIInstrumentor.instrument_app(app)  # auto-instrument all routes

# Structured logging with trace correlation
logger = structlog.get_logger()

@app.get("/orders/{order_id}")
async def get_order(order_id: int, request: Request):
    span = trace.get_current_span()
    trace_id = format(span.get_span_context().trace_id, '032x')

    log = logger.bind(
        trace_id=trace_id,
        order_id=order_id,
        user_id=request.state.user_id
    )

    with tracer.start_as_current_span("db.fetch_order") as db_span:
        db_span.set_attribute("db.order_id", order_id)
        order = await fetch_order(order_id)
        if not order:
            log.warning("order_not_found")
            raise HTTPException(404)

    log.info("order_fetched", status=order.status)
    return order
```

---

# SECTION 7: Pythonic Code & Patterns

**Q26: What are Python's `__init_subclass__` and metaclasses? Show a registry pattern.**

```python
# Registry using __init_subclass__ (cleaner than metaclasses for this)
class EventHandler:
    _registry: dict[str, type] = {}

    def __init_subclass__(cls, event_type: str = "", **kwargs):
        super().__init_subclass__(**kwargs)
        if event_type:
            EventHandler._registry[event_type] = cls

    async def handle(self, event: dict): ...

    @classmethod
    def get_handler(cls, event_type: str) -> 'EventHandler':
        handler_cls = cls._registry.get(event_type)
        if not handler_cls:
            raise ValueError(f"No handler for event type: {event_type}")
        return handler_cls()

class OrderCreatedHandler(EventHandler, event_type="order.created"):
    async def handle(self, event: dict):
        await reserve_inventory(event["order_id"])

class PaymentFailedHandler(EventHandler, event_type="payment.failed"):
    async def handle(self, event: dict):
        await cancel_order(event["order_id"])

# Dispatch incoming events
async def dispatch(event: dict):
    handler = EventHandler.get_handler(event["type"])
    await handler.handle(event)
```

---

**Q27: Context variables for request-scoped state (better than `threading.local` in async).**

```python
from contextvars import ContextVar
from fastapi import Request
import uuid

# Thread-safe AND async-safe: each request/task gets its own copy
request_id_var: ContextVar[str] = ContextVar('request_id', default='')
current_user_var: ContextVar[dict] = ContextVar('current_user', default={})

class ContextMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        token_rid = request_id_var.set(
            request.headers.get("X-Request-ID", str(uuid.uuid4()))
        )
        response = await call_next(request)
        request_id_var.reset(token_rid)
        return response

# Access anywhere without passing through function arguments
def get_current_request_id() -> str:
    return request_id_var.get()

async def some_deep_service_function():
    rid = get_current_request_id()
    logger.info("processing", request_id=rid)
```

---

## Quick-Fire Conceptual Questions

**Q28–Q35 (brief answers expected):**

- **Q28:** What's the difference between `__repr__` and `__str__`? → `__repr__` is for developers (unambiguous), `__str__` for end users (readable). `repr()` falls back to `__repr__` if `__str__` is missing.

- **Q29:** When would you use `weakref`? → When you want a reference to an object without preventing garbage collection — caches, observer patterns, avoiding circular references.

- **Q30:** What is `functools.singledispatch`? → Register different implementations of a function based on the type of the first argument — pythonic way to implement method overloading.

- **Q31:** Difference between `asyncio.Lock` and `threading.Lock`? → `asyncio.Lock` is cooperative (yields to event loop), `threading.Lock` blocks the OS thread. Never use `threading.Lock` in async code.

- **Q32:** What does `__all__` do in a module? → Defines the public API — controls what `from module import *` exports and signals intent to users/IDEs.

- **Q33:** What is `copy_on_write` semantics in Python? → Python doesn't have true COW natively, but assignment never copies — mutation of mutable objects is shared. Use `copy.deepcopy()` or immutable types to avoid surprises.

- **Q34:** How does `functools.lru_cache` handle unhashable arguments? → It raises `TypeError`. Use `functools.cache` (Python 3.9+) for the same but without size limit, or write a custom key function.

- **Q35:** What is the difference between `is` and `==`? → `is` checks identity (same object in memory), `==` checks equality (calls `__eq__`). `None` checks should always use `is None`.

---



# Python & FastAPI Interview Q&A — Extended Set
## Sections 8–14 | Deep Cuts for 5-6 YOE

---

# SECTION 8: Python Type System & Generics

## 8.1 Advanced Typing

**Q36: Explain `TypeVar`, `Generic`, and `ParamSpec`. Build a typed repository base class.**

```python
from typing import TypeVar, Generic, ParamSpec, Callable, Awaitable
from typing import overload
T = TypeVar("T")
ID = TypeVar("ID", int, str)
P = ParamSpec("P")

class Repository(Generic[T, ID]):
    """
    Typed base — subclasses get full IDE autocomplete and mypy checks.
    T  = the domain model type
    ID = the primary key type
    """
    def __init__(self, model_cls: type[T], session):
        self._model = model_cls
        self._session = session

    async def get(self, id: ID) -> T | None:
        return await self._session.get(self._model, id)

    async def list(self, limit: int = 50, offset: int = 0) -> list[T]:
        result = await self._session.execute(
            select(self._model).limit(limit).offset(offset)
        )
        return result.scalars().all()

    async def save(self, entity: T) -> T:
        self._session.add(entity)
        await self._session.flush()
        return entity

    async def delete(self, id: ID) -> None:
        entity = await self.get(id)
        if entity:
            await self._session.delete(entity)

class OrderRepository(Repository[Order, int]):
    # Inherits fully-typed get/list/save/delete
    async def find_by_user(self, user_id: int) -> list[Order]:
        result = await self._session.execute(
            select(Order).where(Order.user_id == user_id)
        )
        return result.scalars().all()

# ParamSpec: preserve callable signatures through decorators
def log_call(fn: Callable[P, Awaitable[T]]) -> Callable[P, Awaitable[T]]:
    async def wrapper(*args: P.args, **kwargs: P.kwargs) -> T:
        logger.info(f"Calling {fn.__name__}", args=args, kwargs=kwargs)
        result = await fn(*args, **kwargs)
        logger.info(f"{fn.__name__} returned", result=result)
        return result
    return wrapper

@log_call
async def create_order(user_id: int, items: list[str]) -> Order:
    ...
# IDE still knows: create_order(user_id: int, items: list[str]) -> Order
```

---

**Q37: What is `Literal`, `TypedDict`, and `Final`? When do you use each over Pydantic?**

```python
from typing import Literal, TypedDict, Final, NotRequired, Required

# Literal: restrict to exact values (better than Enum for simple cases)
Status = Literal["pending", "processing", "shipped", "cancelled"]
HttpMethod = Literal["GET", "POST", "PUT", "PATCH", "DELETE"]

async def update_order_status(order_id: int, status: Status) -> None:
    ...  # mypy rejects update_order_status(1, "delivered") at check time

# TypedDict: typed dicts without class instantiation overhead
# Great for internal plumbing, config dicts, raw JSON shapes
class KafkaMessage(TypedDict):
    topic: str
    partition: int
    offset: int
    key: NotRequired[str]   # optional field (3.11+)
    value: Required[bytes]  # explicitly required

def process_message(msg: KafkaMessage) -> None:
    print(msg["topic"], msg["offset"])   # fully typed, no class overhead

# Final: runtime + type-checker constant — prevents reassignment
MAX_RETRIES: Final = 3
BASE_URL: Final[str] = "https://api.example.com"

MAX_RETRIES = 5  # mypy error: Cannot assign to final name

# When to use TypedDict vs Pydantic:
# TypedDict  → internal code, performance-critical paths, no validation needed
# Pydantic   → API boundaries, user input, needs coercion/validators
```

---

**Q38: How do you use `overload` for a function with multiple call signatures?**

```python
from typing import overload

# Without overload: return type is vague `str | list[str]`
# With overload: caller gets exact type based on how they call it

@overload
async def fetch_user(id: int) -> User: ...
@overload
async def fetch_user(ids: list[int]) -> list[User]: ...

async def fetch_user(id: int | list[int]) -> User | list[User]:
    if isinstance(id, list):
        result = await db.execute(select(User).where(User.id.in_(id)))
        return result.scalars().all()
    return await db.get(User, id)

# Callers get correct types:
single: User = await fetch_user(1)            # User
many: list[User] = await fetch_user([1,2,3])  # list[User]
```

---

# SECTION 9: Concurrency Patterns & Pitfalls

## 9.1 Common Async Bugs

**Q39: What are the most common async bugs in FastAPI services? Show each with a fix.**

```python
# BUG 1: Blocking the event loop with sync I/O
# Bad
@app.get("/bad")
async def bad_endpoint():
    time.sleep(2)        # BLOCKS entire event loop
    data = open("file.txt").read()  # sync I/O blocks too
    return data

# Fix: use asyncio.sleep, aiofiles, or run_in_executor
import aiofiles
@app.get("/good")
async def good_endpoint():
    await asyncio.sleep(2)    # yields to event loop
    async with aiofiles.open("file.txt") as f:
        data = await f.read()
    return data

# BUG 2: Creating tasks without storing references (silently dropped)
# Bad
async def fire_and_forget():
    asyncio.create_task(some_coro())  # task can be GC'd before completion!

# Fix: keep a reference
_background_tasks: set = set()

async def fire_and_forget():
    task = asyncio.create_task(some_coro())
    _background_tasks.add(task)
    task.add_done_callback(_background_tasks.discard)

# BUG 3: Sharing non-thread-safe objects across async tasks
# Bad: SQLAlchemy session shared across concurrent coroutines
session = SessionLocal()
async def bad_parallel():
    await asyncio.gather(
        session.execute(query1),   # same session — race condition!
        session.execute(query2),
    )

# Fix: one session per request / per task
async def good_parallel(pool):
    async with pool() as s1, pool() as s2:
        await asyncio.gather(s1.execute(query1), s2.execute(query2))

# BUG 4: Forgetting to await — no error, just wrong behavior
# Bad
@app.get("/users")
async def get_users(db=Depends(get_db)):
    users = db.execute(select(User))  # returns coroutine, not result!
    return users   # serializes the coroutine object

# Fix
    users = await db.execute(select(User))
```

---

**Q40: Implement a semaphore-controlled connection pool and rate limiter.**

```python
import asyncio
from collections import deque
import time

class RateLimiter:
    """Token bucket algorithm — allows bursts up to `capacity`"""
    def __init__(self, rate: float, capacity: int):
        self.rate = rate           # tokens per second
        self.capacity = capacity
        self._tokens = float(capacity)
        self._last_refill = time.monotonic()
        self._lock = asyncio.Lock()

    async def acquire(self, tokens: int = 1):
        async with self._lock:
            await self._refill()
            if self._tokens < tokens:
                wait = (tokens - self._tokens) / self.rate
                await asyncio.sleep(wait)
                await self._refill()
            self._tokens -= tokens

    async def _refill(self):
        now = time.monotonic()
        elapsed = now - self._last_refill
        self._tokens = min(self.capacity, self._tokens + elapsed * self.rate)
        self._last_refill = now

# Semaphore: bound concurrent outbound calls
class ThirdPartyClient:
    def __init__(self, base_url: str, max_concurrent: int = 10, rps: float = 50):
        self._sem = asyncio.Semaphore(max_concurrent)
        self._limiter = RateLimiter(rate=rps, capacity=rps * 2)
        self._session: aiohttp.ClientSession | None = None

    async def get(self, path: str) -> dict:
        await self._limiter.acquire()
        async with self._sem:   # max 10 in-flight at once
            async with self._session.get(f"{self._base_url}{path}") as resp:
                return await resp.json()
```

---

## 9.2 Structured Concurrency

**Q41: What is structured concurrency and why does it matter in long-lived services?**

```python
# Problem with unstructured concurrency:
# Tasks outlive their logical scope → leaks, zombie tasks, lost exceptions

# Unstructured — task leaks on exception
async def unstructured():
    task = asyncio.create_task(background_job())
    result = await risky_operation()   # raises!
    await task   # never reached → task runs forever

# Structured — TaskGroup cancels all siblings on first failure
async def structured():
    async with asyncio.TaskGroup() as tg:
        t1 = tg.create_task(fetch_user(1))
        t2 = tg.create_task(fetch_orders(1))
        t3 = tg.create_task(fetch_recommendations(1))
    # All three done (or all cancelled on any failure)
    return t1.result(), t2.result(), t3.result()

# Real-world: parallel data enrichment in a FastAPI endpoint
@app.get("/dashboard/{user_id}")
async def dashboard(user_id: int):
    async with asyncio.TaskGroup() as tg:
        user_task    = tg.create_task(get_user(user_id))
        orders_task  = tg.create_task(get_recent_orders(user_id))
        notif_task   = tg.create_task(get_notifications(user_id))

    return {
        "user":          user_task.result(),
        "recent_orders": orders_task.result(),
        "notifications": notif_task.result(),
    }
    # If any fails → all cancel → no partial state leaks
```

---

# SECTION 10: Design Patterns in Python

## 10.1 Creational & Structural

**Q42: Implement the Builder pattern for complex request/query construction.**

```python
from dataclasses import dataclass, field
from typing import Self

@dataclass
class QueryBuilder:
    _table: str = ""
    _conditions: list[str] = field(default_factory=list)
    _order_by: list[str] = field(default_factory=list)
    _limit: int | None = None
    _offset: int = 0
    _joins: list[str] = field(default_factory=list)

    def from_table(self, table: str) -> Self:
        self._table = table
        return self

    def where(self, condition: str) -> Self:
        self._conditions.append(condition)
        return self

    def join(self, table: str, on: str) -> Self:
        self._joins.append(f"JOIN {table} ON {on}")
        return self

    def order(self, column: str, direction: str = "ASC") -> Self:
        self._order_by.append(f"{column} {direction}")
        return self

    def paginate(self, page: int, size: int) -> Self:
        self._limit = size
        self._offset = (page - 1) * size
        return self

    def build(self) -> str:
        sql = f"SELECT * FROM {self._table}"
        if self._joins:
            sql += " " + " ".join(self._joins)
        if self._conditions:
            sql += " WHERE " + " AND ".join(self._conditions)
        if self._order_by:
            sql += " ORDER BY " + ", ".join(self._order_by)
        if self._limit:
            sql += f" LIMIT {self._limit} OFFSET {self._offset}"
        return sql

# Usage — readable, composable, no string spaghetti
query = (
    QueryBuilder()
    .from_table("orders")
    .join("users", "orders.user_id = users.id")
    .where("orders.status = 'pending'")
    .where("users.country = 'US'")
    .order("orders.created_at", "DESC")
    .paginate(page=2, size=25)
    .build()
)
```

---

**Q43: Implement the Strategy pattern for swappable business logic.**

```python
from abc import ABC, abstractmethod
from typing import Protocol

# Strategy via Protocol (more Pythonic than ABC for simple cases)
class PricingStrategy(Protocol):
    def calculate(self, base_price: float, user: dict) -> float: ...

class StandardPricing:
    def calculate(self, base_price: float, user: dict) -> float:
        return base_price

class PremiumPricing:
    def calculate(self, base_price: float, user: dict) -> float:
        return base_price * 0.85  # 15% discount

class B2BPricing:
    def calculate(self, base_price: float, user: dict) -> float:
        volume = user.get("monthly_volume", 1)
        discount = min(0.30, volume * 0.001)  # up to 30%
        return base_price * (1 - discount)

class DynamicPricing:
    def __init__(self, demand_factor: float):
        self.demand_factor = demand_factor

    def calculate(self, base_price: float, user: dict) -> float:
        return base_price * self.demand_factor

PRICING_STRATEGIES: dict[str, PricingStrategy] = {
    "standard": StandardPricing(),
    "premium": PremiumPricing(),
    "b2b": B2BPricing(),
}

class PricingService:
    def get_price(self, product_id: int, user: dict) -> float:
        strategy = PRICING_STRATEGIES.get(user["tier"], StandardPricing())
        base = self._fetch_base_price(product_id)
        return strategy.calculate(base, user)
```

---

**Q44: Chain of Responsibility for request validation pipelines.**

```python
from abc import ABC, abstractmethod

class ValidationHandler(ABC):
    def __init__(self):
        self._next: ValidationHandler | None = None

    def set_next(self, handler: 'ValidationHandler') -> 'ValidationHandler':
        self._next = handler
        return handler   # enables chaining

    async def handle(self, request: dict) -> dict | None:
        if self._next:
            return await self._next.handle(request)
        return request   # passed all checks

class AuthHandler(ValidationHandler):
    async def handle(self, request: dict) -> dict | None:
        if not request.get("token"):
            raise HTTPException(401, "Missing token")
        request["user"] = await verify_token(request["token"])
        return await super().handle(request)

class RateLimitHandler(ValidationHandler):
    async def handle(self, request: dict) -> dict | None:
        user_id = request["user"]["id"]
        if await is_rate_limited(user_id):
            raise HTTPException(429, "Rate limit exceeded")
        return await super().handle(request)

class PermissionHandler(ValidationHandler):
    def __init__(self, required_permission: str):
        super().__init__()
        self.required = required_permission

    async def handle(self, request: dict) -> dict | None:
        user_perms = request["user"].get("permissions", [])
        if self.required not in user_perms:
            raise HTTPException(403, f"Requires {self.required}")
        return await super().handle(request)

# Build the chain
auth = AuthHandler()
rate = RateLimitHandler()
perm = PermissionHandler("orders:write")
auth.set_next(rate).set_next(perm)

@app.post("/orders")
async def create_order(request: Request):
    ctx = {"token": request.headers.get("Authorization"), "body": await request.json()}
    await auth.handle(ctx)
    ...
```

---

# SECTION 11: FastAPI Advanced Patterns

## 11.1 Custom Request Lifecycle

**Q45: How do you implement idempotency keys in FastAPI to make POST endpoints safe to retry?**

```python
import hashlib, json
from fastapi import Header

class IdempotencyMiddleware:
    """Cache responses keyed by Idempotency-Key header"""

    async def __call__(self, request: Request, call_next):
        key = request.headers.get("Idempotency-Key")
        if not key or request.method not in ("POST", "PATCH"):
            return await call_next(request)

        # Include path to namespace keys per endpoint
        cache_key = f"idempotency:{request.url.path}:{key}"
        redis = request.app.state.redis

        cached = await redis.get(cache_key)
        if cached:
            data = json.loads(cached)
            return Response(
                content=data["body"],
                status_code=data["status_code"],
                headers={"X-Idempotent-Replayed": "true",
                         "Content-Type": "application/json"},
            )

        # Lock: prevent concurrent duplicate requests
        lock_key = f"lock:{cache_key}"
        async with redis.lock(lock_key, timeout=30):
            # Double-check after acquiring lock
            cached = await redis.get(cache_key)
            if cached:
                data = json.loads(cached)
                return Response(content=data["body"],
                                status_code=data["status_code"])

            response = await call_next(request)

            # Buffer response body (can only be read once)
            body = b""
            async for chunk in response.body_iterator:
                body += chunk

            await redis.setex(cache_key, 86400, json.dumps({
                "body": body.decode(),
                "status_code": response.status_code,
            }))
            return Response(content=body,
                            status_code=response.status_code,
                            headers=dict(response.headers))
```

---

**Q46: Build a versioned API router with deprecation warnings.**

```python
from fastapi import FastAPI, APIRouter
from fastapi.responses import JSONResponse
from datetime import date

def versioned_router(version: int, deprecated: bool = False,
                     sunset_date: date | None = None) -> APIRouter:
    router = APIRouter(prefix=f"/v{version}")

    if deprecated:
        @router.middleware("http")
        async def deprecation_warning(request: Request, call_next):
            response = await call_next(request)
            response.headers["Deprecation"] = "true"
            response.headers["Sunset"] = str(sunset_date or "")
            response.headers["Link"] = (
                f'</v{version + 1}{request.url.path}>; rel="successor-version"'
            )
            return response

    return router

v1 = versioned_router(version=1, deprecated=True, sunset_date=date(2025, 12, 31))
v2 = versioned_router(version=2)

@v1.get("/users/{id}")
async def get_user_v1(id: int):
    # Old shape — flat structure
    user = await fetch_user(id)
    return {"id": user.id, "name": user.name, "email": user.email}

@v2.get("/users/{id}")
async def get_user_v2(id: int):
    # New shape — nested, richer
    user = await fetch_user(id)
    return {
        "id": user.id,
        "profile": {"name": user.name, "email": user.email},
        "metadata": {"created_at": user.created_at, "role": user.role},
    }

app.include_router(v1)
app.include_router(v2)
```

---

**Q47: Implement a custom response class with envelope pattern and pagination metadata.**

```python
from fastapi.responses import JSONResponse
from fastapi import Request
from math import ceil

class EnvelopedResponse(JSONResponse):
    """Wraps all responses in a consistent envelope"""
    def __init__(self, data, *, status_code=200, message="success", **kwargs):
        content = {"status": "success", "message": message, "data": data}
        super().__init__(content=content, status_code=status_code, **kwargs)

class PaginatedResponse(JSONResponse):
    def __init__(self, data: list, total: int, page: int,
                 size: int, *, status_code=200):
        content = {
            "status": "success",
            "data": data,
            "pagination": {
                "total": total,
                "page": page,
                "size": size,
                "pages": ceil(total / size),
                "has_next": page * size < total,
                "has_prev": page > 1,
            }
        }
        super().__init__(content=content, status_code=status_code)

@app.get("/products")
async def list_products(page: int = 1, size: int = 20, db=Depends(get_db)):
    total = await db.scalar(select(func.count(Product.id)))
    products = await db.execute(
        select(Product).offset((page - 1) * size).limit(size)
    )
    return PaginatedResponse(
        data=[p.__dict__ for p in products.scalars()],
        total=total, page=page, size=size
    )
```

---

## 11.2 Error Handling Architecture

**Q48: Design a structured error hierarchy with FastAPI exception handlers.**

```python
from fastapi import Request
from fastapi.responses import JSONResponse

# Domain error hierarchy
class AppError(Exception):
    status_code: int = 500
    code: str = "internal_error"

    def __init__(self, message: str, details: dict | None = None):
        self.message = message
        self.details = details or {}
        super().__init__(message)

class NotFoundError(AppError):
    status_code = 404
    code = "not_found"

class ConflictError(AppError):
    status_code = 409
    code = "conflict"

class ValidationError(AppError):
    status_code = 422
    code = "validation_error"

class ExternalServiceError(AppError):
    status_code = 502
    code = "external_service_error"

    def __init__(self, service: str, message: str):
        super().__init__(message, {"service": service})

# Register a single handler for the whole hierarchy
@app.exception_handler(AppError)
async def app_error_handler(request: Request, exc: AppError) -> JSONResponse:
    return JSONResponse(
        status_code=exc.status_code,
        content={
            "error": {
                "code": exc.code,
                "message": exc.message,
                "details": exc.details,
                "request_id": getattr(request.state, "request_id", None),
            }
        }
    )

# Usage in service layer — errors bubble up naturally
async def get_order(order_id: int, db) -> Order:
    order = await db.get(Order, order_id)
    if not order:
        raise NotFoundError(f"Order {order_id} not found", {"order_id": order_id})
    return order

async def create_order(data: OrderCreate, db) -> Order:
    existing = await db.scalar(select(Order).where(Order.ref == data.ref))
    if existing:
        raise ConflictError("Order with this ref already exists",
                            {"ref": data.ref, "existing_id": existing.id})
    ...
```

---

# SECTION 12: Data Structures, Algorithms & Python stdlib

**Q49: When would you choose `collections.deque` over a list, and `heapq` over `sorted()`? Show real service use cases.**

```python
from collections import deque
import heapq
import time

# deque: O(1) appendleft/popleft vs O(n) for list
# Use case: sliding window rate limiter
class SlidingWindowLimiter:
    def __init__(self, max_calls: int, window_seconds: float):
        self.max_calls = max_calls
        self.window = window_seconds
        self.calls: deque[float] = deque()   # timestamps

    def is_allowed(self) -> bool:
        now = time.monotonic()
        # Drop timestamps outside window — O(1) popleft
        while self.calls and now - self.calls[0] > self.window:
            self.calls.popleft()
        if len(self.calls) >= self.max_calls:
            return False
        self.calls.append(now)
        return True

# heapq: O(log n) push/pop vs O(n log n) for full re-sort
# Use case: priority job queue in a worker
class PriorityJobQueue:
    def __init__(self):
        self._heap: list[tuple[int, float, dict]] = []
        # (priority, timestamp, job) — timestamp breaks ties

    def push(self, job: dict, priority: int):
        heapq.heappush(self._heap, (priority, time.monotonic(), job))

    def pop(self) -> dict:
        _, _, job = heapq.heappop(self._heap)
        return job

    def peek_priority(self) -> int | None:
        return self._heap[0][0] if self._heap else None

queue = PriorityJobQueue()
queue.push({"type": "report"}, priority=5)   # lower = higher priority
queue.push({"type": "email"}, priority=1)
queue.push({"type": "invoice"}, priority=3)
print(queue.pop())  # {"type": "email"}  — priority 1 first
```

---

**Q50: `collections.defaultdict`, `Counter`, and `ChainMap` — real use cases.**

```python
from collections import defaultdict, Counter, ChainMap

# defaultdict: eliminate key-existence checks
# Use: grouping events by type in a stream processor
def group_events(events: list[dict]) -> dict[str, list]:
    grouped: defaultdict[str, list] = defaultdict(list)
    for event in events:
        grouped[event["type"]].append(event)
    return dict(grouped)

# Counter: counting with math operations
# Use: analyze API endpoint hit frequencies
from collections import Counter

def analyze_access_log(log_entries: list[str]) -> dict:
    counter = Counter(log_entries)
    return {
        "top_10": counter.most_common(10),
        "total_unique": len(counter),
        "total_requests": sum(counter.values()),
    }

# Counter arithmetic
yesterday = Counter({"GET /api/users": 500, "POST /api/orders": 100})
today     = Counter({"GET /api/users": 600, "POST /api/orders": 80})
delta = today - yesterday     # only positive differences
print(delta)  # Counter({'GET /api/users': 100})

# ChainMap: layered config (environment overrides defaults)
# Use: config resolution chain
import os

defaults    = {"DEBUG": False, "DB_POOL_SIZE": 5, "CACHE_TTL": 300}
from_file   = {"DB_POOL_SIZE": 10, "LOG_LEVEL": "INFO"}
from_env    = {k: v for k, v in os.environ.items() if k in defaults}

config = ChainMap(from_env, from_file, defaults)
# Resolution: env vars > file config > defaults
print(config["DB_POOL_SIZE"])  # env if set, else 10, else 5
```

---

**Q51: How does Python's `bisect` module work? Use it to implement tiered pricing lookup in O(log n).**

```python
import bisect

class TieredPricingEngine:
    """
    Price tiers defined by volume thresholds.
    Lookup must be O(log n) — called millions of times per day.
    """
    def __init__(self, tiers: list[tuple[int, float]]):
        # tiers: [(min_volume, price_per_unit), ...] sorted by volume
        tiers_sorted = sorted(tiers, key=lambda t: t[0])
        self._thresholds = [t[0] for t in tiers_sorted]
        self._prices = [t[1] for t in tiers_sorted]

    def get_price(self, volume: int) -> float:
        # bisect_right: find rightmost position where threshold <= volume
        idx = bisect.bisect_right(self._thresholds, volume) - 1
        idx = max(0, idx)
        return self._prices[idx]

    def get_total(self, volume: int) -> float:
        return volume * self.get_price(volume)

engine = TieredPricingEngine(tiers=[
    (0,    1.00),   # 0-99 units: $1.00 each
    (100,  0.85),   # 100-499: $0.85
    (500,  0.70),   # 500-999: $0.70
    (1000, 0.55),   # 1000+: $0.55
])

print(engine.get_price(50))    # 1.00
print(engine.get_price(200))   # 0.85
print(engine.get_price(5000))  # 0.55
```

---

# SECTION 13: Security Patterns

**Q52: Implement secure JWT handling with refresh token rotation.**

```python
from datetime import datetime, timedelta, timezone
from jose import jwt, JWTError
from passlib.context import CryptContext
import secrets

pwd_ctx = CryptContext(schemes=["bcrypt"], deprecated="auto")

class TokenService:
    ACCESS_EXPIRE  = timedelta(minutes=15)
    REFRESH_EXPIRE = timedelta(days=30)
    ALGORITHM = "HS256"

    def __init__(self, secret: str, redis):
        self.secret = secret
        self.redis = redis

    def create_access_token(self, user_id: int, role: str) -> str:
        return jwt.encode({
            "sub": str(user_id),
            "role": role,
            "type": "access",
            "exp": datetime.now(timezone.utc) + self.ACCESS_EXPIRE,
            "iat": datetime.now(timezone.utc),
        }, self.secret, algorithm=self.ALGORITHM)

    async def create_refresh_token(self, user_id: int) -> str:
        token = secrets.token_urlsafe(64)
        await self.redis.setex(
            f"refresh:{token}", int(self.REFRESH_EXPIRE.total_seconds()), str(user_id)
        )
        return token

    async def rotate_refresh_token(self, old_token: str) -> tuple[str, str]:
        """Rotation: old token consumed, new pair issued. Replay = revoke all."""
        user_id_bytes = await self.redis.getdel(f"refresh:{old_token}")
        if not user_id_bytes:
            # Possible replay attack — revoke all sessions for safety
            await self._revoke_all_sessions_for_token(old_token)
            raise HTTPException(401, "Invalid or replayed refresh token")

        user_id = int(user_id_bytes)
        user = await get_user(user_id)
        access  = self.create_access_token(user_id, user.role)
        refresh = await self.create_refresh_token(user_id)
        return access, refresh

    def decode_access_token(self, token: str) -> dict:
        try:
            payload = jwt.decode(token, self.secret, algorithms=[self.ALGORITHM])
            if payload.get("type") != "access":
                raise HTTPException(401, "Wrong token type")
            return payload
        except JWTError:
            raise HTTPException(401, "Invalid token")
```

---

**Q53: SQL injection, mass assignment, and SSRF — show each vulnerability and its fix in a FastAPI context.**

```python
# VULNERABILITY 1: SQL Injection
# Bad
@app.get("/users")
async def bad_search(name: str, db=Depends(get_db)):
    query = f"SELECT * FROM users WHERE name = '{name}'"  # injectable!
    return await db.execute(query)   # name = "'; DROP TABLE users;--"

# Fix: parameterized queries always
@app.get("/users")
async def safe_search(name: str, db=Depends(get_db)):
    result = await db.execute(select(User).where(User.name == name))
    return result.scalars().all()

# VULNERABILITY 2: Mass assignment
# Bad — user controls which fields get written
@app.patch("/users/{id}")
async def bad_update(id: int, data: dict, db=Depends(get_db)):
    user = await db.get(User, id)
    for key, val in data.items():
        setattr(user, key, val)   # user can set is_admin=True, role="admin"!

# Fix: explicit Pydantic schema with only allowed fields
class UserUpdate(BaseModel):
    name: str | None = None
    email: str | None = None
    # is_admin, role, etc. intentionally excluded

@app.patch("/users/{id}")
async def safe_update(id: int, data: UserUpdate, db=Depends(get_db)):
    user = await db.get(User, id)
    for field, value in data.model_dump(exclude_unset=True).items():
        setattr(user, field, value)

# VULNERABILITY 3: SSRF (Server-Side Request Forgery)
import ipaddress
from urllib.parse import urlparse

ALLOWED_SCHEMES = {"https"}
BLOCKED_HOSTS = {"169.254.169.254", "metadata.google.internal"}  # cloud metadata

def validate_url(url: str) -> str:
    parsed = urlparse(url)
    if parsed.scheme not in ALLOWED_SCHEMES:
        raise ValueError("Only HTTPS URLs allowed")
    host = parsed.hostname or ""
    if host in BLOCKED_HOSTS:
        raise ValueError("Blocked host")
    try:
        ip = ipaddress.ip_address(host)
        if ip.is_private or ip.is_loopback or ip.is_link_local:
            raise ValueError("Private/internal IP not allowed")
    except ValueError:
        pass   # not an IP — domain, allow
    return url

@app.post("/webhook/test")
async def test_webhook(url: str):
    safe_url = validate_url(url)
    async with httpx.AsyncClient() as client:
        resp = await client.post(safe_url, json={"test": True})
    return {"status": resp.status_code}
```

---

# SECTION 14: Deployment, Config & Observability

**Q54: Twelve-factor app config with Pydantic Settings and secret management.**

```python
from pydantic_settings import BaseSettings, SettingsConfigDict
from pydantic import SecretStr, field_validator, AnyUrl
from functools import lru_cache

class DatabaseSettings(BaseSettings):
    host: str = "localhost"
    port: int = 5432
    name: str = "app"
    user: str = "postgres"
    password: SecretStr   # never logged, never in repr()

    @property
    def url(self) -> str:
        pwd = self.password.get_secret_value()
        return f"postgresql+asyncpg://{self.user}:{pwd}@{self.host}:{self.port}/{self.name}"

class Settings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        env_nested_delimiter="__",   # DB__HOST=localhost maps to db.host
        case_sensitive=False,
    )

    app_name: str = "my-service"
    environment: str = "development"
    debug: bool = False
    db: DatabaseSettings = DatabaseSettings()
    redis_url: AnyUrl = "redis://localhost:6379"
    secret_key: SecretStr
    allowed_origins: list[str] = ["http://localhost:3000"]
    workers: int = 1

    @field_validator("environment")
    @classmethod
    def must_be_valid_env(cls, v: str) -> str:
        if v not in ("development", "staging", "production"):
            raise ValueError(f"Invalid environment: {v}")
        return v

    @property
    def is_production(self) -> bool:
        return self.environment == "production"

@lru_cache  # settings loaded once per process
def get_settings() -> Settings:
    return Settings()

# FastAPI DI
@app.get("/health")
async def health(settings: Settings = Depends(get_settings)):
    return {"env": settings.environment, "app": settings.app_name}
```

---

**Q55: Implement health checks with dependency probing for Kubernetes liveness/readiness.**

```python
import asyncio
from enum import Enum

class HealthStatus(str, Enum):
    HEALTHY   = "healthy"
    DEGRADED  = "degraded"
    UNHEALTHY = "unhealthy"

async def check_postgres(db_pool) -> dict:
    try:
        async with asyncio.timeout(2.0):
            await db_pool.execute("SELECT 1")
        return {"status": HealthStatus.HEALTHY, "latency_ms": 0}
    except asyncio.TimeoutError:
        return {"status": HealthStatus.UNHEALTHY, "error": "timeout"}
    except Exception as e:
        return {"status": HealthStatus.UNHEALTHY, "error": str(e)}

async def check_redis(redis) -> dict:
    try:
        async with asyncio.timeout(1.0):
            await redis.ping()
        return {"status": HealthStatus.HEALTHY}
    except Exception as e:
        return {"status": HealthStatus.UNHEALTHY, "error": str(e)}

# /livez — is the process alive? (k8s kills + restarts on failure)
@app.get("/livez", status_code=200)
async def liveness():
    return {"status": "alive"}

# /readyz — is the process ready to serve traffic? (k8s holds traffic on failure)
@app.get("/readyz")
async def readiness(request: Request):
    checks = await asyncio.gather(
        check_postgres(request.app.state.db_pool),
        check_redis(request.app.state.redis),
        return_exceptions=True
    )
    results = {
        "postgres": checks[0] if not isinstance(checks[0], Exception) else {"status": "unhealthy"},
        "redis":    checks[1] if not isinstance(checks[1], Exception) else {"status": "unhealthy"},
    }
    all_healthy = all(c["status"] == HealthStatus.HEALTHY for c in results.values())
    any_unhealthy = any(c["status"] == HealthStatus.UNHEALTHY for c in results.values())

    overall = (HealthStatus.HEALTHY if all_healthy
               else HealthStatus.UNHEALTHY if any_unhealthy
               else HealthStatus.DEGRADED)

    return JSONResponse(
        status_code=200 if overall != HealthStatus.UNHEALTHY else 503,
        content={"status": overall, "checks": results}
    )
```

---

## More Quick-Fire Q&A (Q56–Q65)

**Q56:** What is `__future__` annotations and why use it?
→ `from __future__ import annotations` makes all annotations strings (lazy evaluation) — avoids forward reference errors and speeds up module import. In Python 3.10+ it's partially the default.

**Q57:** Difference between `os.environ` and `dotenv` loading?
→ `os.environ` reads the current process environment. `dotenv` reads a `.env` file and injects into `os.environ` — useful locally but should never override already-set env vars in production (`override=False` default).

**Q58:** What is `__class_getitem__` used for?
→ Enables `MyClass[T]` syntax without inheriting `Generic`. Used to create generic aliases, e.g., `list[int]` in stdlib works because `list.__class_getitem__(int)` returns `list[int]`.

**Q59:** When does `@staticmethod` make more sense than a module-level function?
→ When the logic belongs conceptually to the class namespace but doesn't need `self` or `cls` — e.g., a `User.validate_email(email)` factory helper. Module-level functions are fine when the logic is genuinely standalone.

**Q60:** What's wrong with using mutable defaults in function signatures?
```python
# Bug: all callers share the SAME list object
def add_item(item, items=[]):
    items.append(item)
    return items

add_item("a")  # ["a"]
add_item("b")  # ["a", "b"]  ← surprise!

# Fix:
def add_item(item, items=None):
    if items is None:
        items = []
    items.append(item)
    return items
```

**Q61:** What is `__enter__`/`__exit__` vs `__aenter__`/`__aexit__`?
→ Sync vs async context manager protocols. FastAPI dependencies using `yield` with `async with` require `__aenter__`/`__aexit__`. Always prefer `@contextmanager` / `@asynccontextmanager` over implementing dunder methods manually.

**Q62:** How does `functools.partial` differ from a lambda?
→ `partial` is more explicit, picklable (works with `multiprocessing`), and plays well with `inspect`. Lambdas are inline, not picklable, and harder to test/debug. Use `partial` for wiring up callbacks and executor tasks.

**Q63:** What is the `__call__` protocol and how does FastAPI use it?
→ Any object with `__call__` is callable. FastAPI uses this for class-based dependencies — `Depends(MyClass())` works because `MyClass.__call__` is invoked per request, enabling stateful per-class setup with per-call logic.

**Q64:** Difference between `pickle`, `json`, and `msgpack` for inter-service serialization?
→ `pickle` is Python-only, fast, supports arbitrary objects — never use for untrusted input. `json` is universal, human-readable, slow for large payloads. `msgpack` is binary, ~2-5x faster than JSON, schema-less but not human-readable — good for internal Kafka/Redis payloads.

**Q65:** What is `sys.intern` and when does it matter?
→ Forces string deduplication — two interned strings with same value share memory. Matters when you have millions of short repeated strings (event type names, status codes, tags). FastAPI/Pydantic field names are auto-interned by Python.

---



> **Coverage summary across both sets:** Python internals, async/concurrency, type system, Pydantic v2, DI patterns, REST/GraphQL/gRPC/WebSocket/EDA/CQRS, testing strategy, SQLAlchemy, security, design patterns, stdlib data structures, config management, and observability. That's roughly 65 questions across 14 sections — solid prep for a senior backend Python interview.